In [1]:
!pip install -q transformers==4.46.0 peft bitsandbytes accelerate jamotools jamo trl -q

In [2]:
import os
import io
import gc
import base64
import torch
import torchaudio
import jamotools
from jamo import h2j, j2hcj
from IPython.display import display, Javascript
from google.colab import output, drive
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, AutoModelForSequenceClassification
)
from peft import PeftModel

# 1. Mount Storage
drive.mount('/content/drive')

# 2. Performance Tracking Hardware Verification
print(f"✅ GPU Active: {torch.cuda.is_available()}")
print(f"🖥️ Execution Device: {torch.cuda.get_device_name(0)}")

# 3. SET GLOBALS: Points directly to your finalized model assets in Drive
SAVE_DIR               = "/content/drive/MyDrive/manual_datasets/dialogue_system2"
ASR_MODEL_PATH         = "/content/drive/MyDrive/manual_datasets/clovacall_data/final_asr_jamo_model"
ROUTER_PATH            = os.path.join(SAVE_DIR, "intent_router")
RESTAURANT_EXPERT_PATH = os.path.join(SAVE_DIR, "restaurant_expert")
TRAVEL_EXPERT_PATH     = os.path.join(SAVE_DIR, "travel_expert")
SLM_MODEL_ID           = "Qwen/Qwen2.5-1.5B-Instruct"

print("✅ Configuration paths mapped successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GPU Active: True
🖥️ Execution Device: Tesla T4
✅ Configuration paths mapped successfully!


In [3]:
def record_user_voice(filename="my_practice.wav"):
    """Spins up HTML5 audio interface to record speech samples natively inside Colab."""
    js = Javascript("""
    async function recordAudio() {
      const div = document.createElement('div');
      const btn = document.createElement('button');
      const str = document.createElement('span');

      btn.textContent = '🎤 Click to Start Recording';
      btn.style.cssText = "padding:10px; background:#f44336; color:white; border:none; border-radius:5px; cursor:pointer; font-size:16px; margin:10px;";

      document.body.appendChild(div);
      div.appendChild(btn);
      div.appendChild(str);

      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      let chunks = [];

      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.onstop = async () => {
        const blob = new Blob(chunks, { type: 'audio/wav' });
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          window.audioData = reader.result.split(',')[1];
        };
      };

      btn.onclick = () => {
        if (recorder.state === 'inactive') {
          recorder.start();
          btn.textContent = '🛑 Stop Recording';
          btn.style.background = '#2196F3';
        } else {
          recorder.stop();
          btn.textContent = '✅ Processing Voice Matrix...';
          btn.disabled = true;
        }
      };

      while (recorder.state !== 'inactive' || !window.audioData) {
        await new Promise(resolve => setTimeout(resolve, 100));
      }

      const data = window.audioData;
      window.audioData = null;
      div.remove();
      return data;
    }
    window.recordAudio = recordAudio;
    """)

    display(js)
    print("🎤 Listening for voice input...")
    audio_data_base64 = output.eval_js('window.recordAudio()')

    with open(filename, "wb") as f:
        f.write(base64.b64decode(audio_data_base64))

    return filename

In [4]:
print("📦 Loading Acoustic Signal Processors...")
asr_processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_PATH)
asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_PATH).to("cuda")
asr_model.eval()

print("🧭 Loading Gating Intent Router...")
# 🔄 FIXED: Pull clean tokenizer files from the source repo, but keep loading your custom local weights model!
router_tokenizer = AutoTokenizer.from_pretrained("klue/roberta-small")
router_model = AutoModelForSequenceClassification.from_pretrained(ROUTER_PATH).to("cuda")
router_model.eval()

print("✅ Static background scoring frameworks loaded completely!")

📦 Loading Acoustic Signal Processors...
🧭 Loading Gating Intent Router...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Static background scoring frameworks loaded completely!


In [5]:
def compute_edit_distance(ref, hyp):
    N, M = len(ref), len(hyp)
    dp = [[0] * (M + 1) for _ in range(N + 1)]
    for i in range(N + 1): dp[i][0] = i
    for j in range(M + 1): dp[0][j] = j
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    return dp

def backtrack(dp, ref, hyp):
    i, j = len(ref), len(hyp)
    operations = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            operations.append({"position": i, "type": "substitution", "expected": ref[i-1], "predicted": hyp[j-1]})
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            operations.append({"position": i, "type": "deletion", "expected": ref[i-1], "predicted": "[missing]"})
            i -= 1;
        else:
            operations.append({"position": j, "type": "insertion", "expected": "[none]", "predicted": hyp[j-1]})
            j -= 1
    operations.reverse()
    return operations

def get_pronunciation_diagnostics(audio_file, target_hangul):
    speech, sr = torchaudio.load(audio_file)
    if sr != 16000:
        speech = torchaudio.transforms.Resample(sr, 16000)(speech)

    input_values = asr_processor(speech.squeeze().numpy(), sampling_rate=16000, return_tensors="pt").input_values.to("cuda")
    with torch.no_grad():
        logits = asr_model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_jamo_str = asr_processor.batch_decode(pred_ids)[0]
    transcription = jamotools.join_jamos(pred_jamo_str).strip()

    ref_clean, hyp_clean = target_hangul.replace(" ", ""), transcription.replace(" ", "")
    ref_jamo, hyp_jamo = list(jamotools.split_syllables(ref_clean)), list(jamotools.split_syllables(hyp_clean))

    dp = compute_edit_distance(ref_jamo, hyp_jamo)
    per = (dp[len(ref_jamo)][len(hyp_jamo)] / len(ref_jamo)) if ref_jamo else 0.0

    ops = backtrack(dp, ref_jamo, hyp_jamo)
    s = sum(1 for o in ops if o["type"] == "substitution")
    d = sum(1 for o in ops if o["type"] == "deletion")
    i = sum(1 for o in ops if o["type"] == "insertion")

    syl_dp = compute_edit_distance(list(ref_clean), list(hyp_clean))
    syl_errors = backtrack(syl_dp, list(ref_clean), list(hyp_clean))

    return transcription, per, syl_errors, (s, d, i)

def route_intent(text):
    """Routes the user transcription to the correct domain expert adapter safely."""
    encoding = router_tokenizer(
        text,
        max_length=64,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids      = encoding['input_ids'].to("cuda")
    attention_mask = encoding['attention_mask'].to("cuda")

    with torch.no_grad():
        outputs = router_model(input_ids=input_ids, attention_mask=attention_mask)

    # 🔄 FIXED: Check if outputs is a dictionary or a tuple to prevent AttributeError
    if hasattr(outputs, "logits"):
        logits = outputs.logits
    else:
        logits = outputs[0]  # If it returns a raw tuple, logits sit at index 0

    probs = torch.softmax(logits, dim=-1)
    pred  = torch.argmax(probs, dim=-1).item()

    return "restaurant" if pred == 0 else "travel"

In [17]:
def load_expert(expert_path):
    """Dynamic model loader implementing strict garbage collection to safeguard VRAM boundaries."""
    global expert_model
    if 'expert_model' in globals():
        del expert_model
    gc.collect()
    torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        SLM_MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
    return PeftModel.from_pretrained(base_model, expert_path)

def generate_tutor_response(transcription, per, syl_errors, error_counts, target_sentence):
    intent = route_intent(transcription)
    print(f"🧭 Gating Node Target Match: [{intent.upper()} EXPERT]")

    expert_path   = RESTAURANT_EXPERT_PATH if intent == "restaurant" else TRAVEL_EXPERT_PATH
    expert_model  = load_expert(expert_path)
    slm_tokenizer = AutoTokenizer.from_pretrained(SLM_MODEL_ID, trust_remote_code=True)

    s, d, i = error_counts
    role = "restaurant host in Seoul" if intent == "restaurant" else "tour guide in Northern Seoul"

    # Build error string from actual Python data — no hallucination possible
    feedback_sentence = "Your pronunciation was excellent and clear!"
    if syl_errors:
        sub_errors = [err for err in syl_errors if err.get("type") == "substitution"]
        if sub_errors:
            details = [f"'{err['predicted']}' instead of '{err['expected']}'" for err in sub_errors]
            feedback_sentence = "You accidentally said " + " and ".join(details) + "."

    # Separate prompt for perfect vs imperfect pronunciation
    if per == 0.0:
        system_instruction = (
            f"You are an expert AI Korean Pronunciation Coach roleplaying as a {role}. Speak ONLY in English.\n\n"
            f"SITUATION: A Korean language student just spoke to you perfectly.\n"
            f"WHAT THEY SAID: '{transcription}'\n\n"
            f"YOUR TASK:\n"
            f"1. In 1 sentence, praise their perfect pronunciation.\n"
            f"2. In 1 sentence, respond in character to their request.\n"
            f"IMPORTANT: Do NOT mention any errors. There were none. Do not invent corrections."
        )
    else:
        system_instruction = (
            f"You are an expert AI Korean Pronunciation Coach roleplaying as a {role}. Speak ONLY in English.\n\n"
            f"SITUATION: A Korean language student just spoke to you.\n"
            f"WHAT THEY WERE SUPPOSED TO SAY: '{target_sentence}'\n"
            f"WHAT YOU HEARD: '{transcription}'\n"
            f"PRONUNCIATION FEEDBACK: {feedback_sentence}\n\n"
            f"YOUR TASK:\n"
            f"1. In 1 sentence, gently tell the student which sound they got wrong using the PRONUNCIATION FEEDBACK above.\n"
            f"2. In 1 sentence, respond in character to their request.\n"
            f"Do not apologize. Do not pretend to be the student. You are the coach and {role.split()[0]}."
        )

    user_payload = f"Target: {target_sentence}\nTranscription: {transcription}\nGenerate the response now."

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user",   "content": user_payload}
    ]
    prompt = slm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = slm_tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = expert_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=0.1,
            top_p=0.9,
            pad_token_id=slm_tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return slm_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

In [21]:
# 🎯 1. Input what you want the student to say
target_sentence = "물 좀 주세요."
print(f"🎯 Practice Objective: {target_sentence}\n" + "-"*50)

# 🎤 2. Run browser recording
audio_input = record_user_voice("live_test.wav")

# 📡 3. Transcribe speech, calculate Phoneme Error Rate (PER), route intent, and run MoE generation
print("\n📡 Processing engine running...")
transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(audio_input, target_sentence)

print("\n🤖 Querying expert adapters...")
response = generate_tutor_response(transcription, per, syl_errors, error_counts, target_sentence)

print("=" * 60)
print(f"📝 Expected Statement:   {target_sentence}")
print(f"📝 Heard Statement:      {transcription}")
print(f"📊 Phoneme Accuracy:     {(1.0 - per)*100:.1f}% Accurate")
print(f"🤖 Language Coach Feedback:\n{response}")
print("=" * 60)

🎯 Practice Objective: 물 좀 주세요.
--------------------------------------------------


<IPython.core.display.Javascript object>

🎤 Listening for voice input...

📡 Processing engine running...

🤖 Querying expert adapters...
🧭 Gating Node Target Match: [RESTAURANT EXPERT]
📝 Expected Statement:   물 좀 주세요.
📝 Heard Statement:      물좀 주세요
📊 Phoneme Accuracy:     92.3% Accurate
🤖 Language Coach Feedback:
Your pronunciation is perfect! Please let me know how much water you would like. I will get it for you right away.


In [22]:
# 🎯 1. Input what you want the student to say
target_sentence = "여기 명소 추천해주세요."
print(f"🎯 Practice Objective: {target_sentence}\n" + "-"*50)

# 🎤 2. Run browser recording
audio_input = record_user_voice("live_test.wav")

# 📡 3. Transcribe speech, calculate Phoneme Error Rate (PER), route intent, and run MoE generation
print("\n📡 Processing engine running...")
transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(audio_input, target_sentence)

print("\n🤖 Querying expert adapters...")
response = generate_tutor_response(transcription, per, syl_errors, error_counts, target_sentence)

print("=" * 60)
print(f"📝 Expected Statement:   {target_sentence}")
print(f"📝 Heard Statement:      {transcription}")
print(f"📊 Phoneme Accuracy:     {(1.0 - per)*100:.1f}% Accurate")
print(f"🤖 Language Coach Feedback:\n{response}")
print("=" * 60)

🎯 Practice Objective: 여기 명소 추천해주세요.
--------------------------------------------------


<IPython.core.display.Javascript object>

🎤 Listening for voice input...

📡 Processing engine running...

🤖 Querying expert adapters...
🧭 Gating Node Target Match: [TRAVEL EXPERT]
📝 Expected Statement:   여기 명소 추천해주세요.
📝 Heard Statement:      여기 명서 추천해 주세요.
📊 Phoneme Accuracy:     95.7% Accurate
🤖 Language Coach Feedback:
Your pronunciation is correct. I recommend visiting Namsangol Hanok Village. It's located in Gangnam District.


In [24]:
# 🎯 1. Input what you want the student to say
target_sentence = "비빔밥 하나 주세요"
print(f"🎯 Practice Objective: {target_sentence}\n" + "-"*50)

# 🎤 2. Run browser recording
audio_input = record_user_voice("live_test.wav")

# 📡 3. Transcribe speech, calculate Phoneme Error Rate (PER), route intent, and run MoE generation
print("\n📡 Processing engine running...")
transcription, per, syl_errors, error_counts = get_pronunciation_diagnostics(audio_input, target_sentence)

print("\n🤖 Querying expert adapters...")
response = generate_tutor_response(transcription, per, syl_errors, error_counts, target_sentence)

print("=" * 60)
print(f"📝 Expected Statement:   {target_sentence}")
print(f"📝 Heard Statement:      {transcription}")
print(f"📊 Phoneme Accuracy:     {(1.0 - per)*100:.1f}% Accurate")
print(f"🤖 Language Coach Feedback:\n{response}")
print("=" * 60)

🎯 Practice Objective: 비빔밥 하나 주세요
--------------------------------------------------


<IPython.core.display.Javascript object>

🎤 Listening for voice input...

📡 Processing engine running...

🤖 Querying expert adapters...
🧭 Gating Node Target Match: [RESTAURANT EXPERT]
📝 Expected Statement:   비빔밥 하나 주세요
📝 Heard Statement:      미빔밥 하나 주세요.
📊 Phoneme Accuracy:     88.9% Accurate
🤖 Language Coach Feedback:
Your pronunciation is almost correct. The missing vowel makes it '비' instead of '미'. Let's try again.
